In [ ]:
import laser_lib
import numpy as np
import math
import time
#import cv2
#from skimage.color import rgb2lab, lab2rgb
queue = laser_lib.DacQueue()

In [ ]:
# Circle
queue.dac_rate = 30000
T = 500
arr_pos = np.zeros((T, 2))
arr_col = np.zeros((T, 3))
arr_col[:, :] = 1
theta = np.linspace(0, 2*np.pi, T)
arr_pos[:, 0] = np.cos(theta)
arr_pos[:, 1] = np.sin(theta)

#arr_col = arr_col*np.expand_dims((theta > np.pi/2), 1)

while(True):
    queue.submit(arr_pos, arr_col, loop=False)
queue.submit(arr_pos, np.zeros((T,3)), loop=True)

In [ ]:
import numpy as np
import time
import cv2

# --- Face Detection Setup ---
# Use the CPU-based Haar Cascade
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)
cap = cv2.VideoCapture(0)

# Settings
queue.dac_rate = 30000
points_per_unit = 400
r_outer, r_inner = 0.4, 0.1
brow_w, brow_h = 0.4, 0.15
transition_points = 20
laser_offset = -1
overshoot = 0.0275
offset_distance = 0.6
pupil_height = r_outer * 0.45

# Blink Settings
blink_interval = 4.0   
blink_duration = 0.2   
brow_drop = 0.15       

# Tracking Settings
max_angle = np.deg2rad(38) # How far the eyes can rotate
current_x, current_y = 0.0, 0.0
target_x, target_y = 0.0, 0.0
lerp_speed = 0.2  # Smooths out jitter from the webcam detection

# --- 1. Generate Local Geometry ---
t_out = int(points_per_unit * r_outer)
t_in = int(points_per_unit * r_inner)

theta_out = np.linspace(-overshoot * 2*np.pi, 2*np.pi + 2*np.pi * overshoot, t_out)
pts_out = np.column_stack([r_outer * np.cos(theta_out), r_outer * np.sin(theta_out), np.zeros(t_out)])

theta_in = np.linspace(-overshoot * 2*np.pi, 2*np.pi + 2*np.pi * overshoot, t_in)
pts_in = np.column_stack([r_inner * np.cos(theta_in), r_inner * np.sin(theta_in), pupil_height * np.ones(t_in)])

trans_out_to_in = np.linspace(pts_out[-1], pts_in[0], transition_points)
local_pair = np.vstack([pts_out, trans_out_to_in, pts_in])

t_brow = int(points_per_unit * brow_w)
theta_brow = np.linspace(0.2, 0.8 * np.pi, t_brow)
pts_brow = np.column_stack([brow_w * np.cos(theta_brow), 0.6 + brow_h * np.sin(theta_brow), np.zeros(t_brow)])

# --- 2. Main Loop ---
while True:
    t = time.time()
    

    # --- 2a. Face Detection & Target Calculation ---
    ret, frame = cap.read()
    if ret:
        # Flip frame so the eyes "mirror" the user
        frame = cv2.flip(frame, 1)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))

        if len(faces) > 0:
            # Find largest face
            (fx, fy, fw, fh) = max(faces, key=lambda r: r[2] * r[3])
            
            # Find center of the face
            face_center_x = fx + (fw / 2)
            face_center_y = fy + (fh / 2)

            # --- DRAWING SECTION ---
            # Draw the bounding box for the largest face (Green)
            cv2.rectangle(frame, (fx, fy), (fx + fw, fy + fh), (0, 255, 0), 2)
            
            # Draw a small dot at the tracking center (Red)
            cv2.circle(frame, (int(face_center_x), int(face_center_y)), 5, (0, 0, 255), -1)
            
            # Label the face
            cv2.putText(frame, "Tracking Primary Face", (fx, fy - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            # ------------------------

            # Normalize position to range [-1.0, 1.0]
            norm_x = (face_center_x / frame.shape[1]) * 2 - 1
            norm_y = (face_center_y / frame.shape[0]) * 2 - 1

            # Map normalized position to radians
            target_y = norm_x * max_angle 
            target_x = norm_y * max_angle 
        
        # Display the webcam preview with the boxes
        cv2.imshow('Laser Eye Tracking Debug', frame)
        
        # Keep waitKey(1) to allow the window to refresh
        if cv2.waitKey(1) & 0xFF == ord('q'): 
            break

    # Smoothly move eyes to target
    current_x += (target_x - current_x) * lerp_speed
    current_y += (target_y - current_y) * lerp_speed

    cx, sx = np.cos(current_x), np.sin(current_x)
    cy, sy = np.cos(current_y), np.sin(current_y)
    rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    R = (ry @ rx).T

    rotated_pair = local_pair @ R

    # --- 3. Blink Logic ---
    t_mod = t % blink_interval
    if t_mod < blink_duration:
        phase = t_mod / blink_duration
        blink_amt = np.sin(phase * np.pi) 
    else:
        blink_amt = 0.0
        
    eyelid_y = r_outer - (r_outer * 2 * blink_amt)
    rotated_pair[:, 1] = np.minimum(rotated_pair[:, 1], eyelid_y)
    current_brow_drop = brow_drop * blink_amt

    # --- 4. Assembly ---
    eye_l_pupil = rotated_pair.copy() - [offset_distance, 0, 0]
    brow_l = pts_brow.copy() - [offset_distance, current_brow_drop, 0]

    eye_r_pupil = rotated_pair.copy() + [offset_distance, 0, 0]
    brow_r = pts_brow.copy()
    brow_r[:, 0] *= -1 
    brow_r[:, 0] += offset_distance
    brow_r[:, 1] -= current_brow_drop 

    t_l_to_brow = np.linspace(eye_l_pupil[-1], brow_l[0], transition_points)
    t_brow_to_r = np.linspace(brow_l[-1], eye_r_pupil[0], transition_points)
    t_r_to_brow = np.linspace(eye_r_pupil[-1], brow_r[0], transition_points)
    t_wrap = np.linspace(brow_r[-1], eye_l_pupil[0], transition_points)

    full_frame_3d = np.vstack([
        eye_l_pupil, t_l_to_brow, brow_l, t_brow_to_r,
        eye_r_pupil, t_r_to_brow, brow_r, t_wrap
    ])

    # --- 5. Color Mapping ---
    arr_col = np.zeros((len(full_frame_3d), 3))
    curr = 0
    for _ in range(2): 
        arr_col[curr + 3 : curr + t_out - 3, :] = 1
        curr += t_out + transition_points 
        arr_col[curr + 3 : curr + t_in - 3, :] = 1
        curr += t_in + transition_points
        arr_col[curr + 3 : curr + t_brow - 3, :] = 1
        if _ == 0: curr += t_brow + transition_points

    arr_col = np.roll(arr_col, -laser_offset, axis=0)
    queue.submit(full_frame_3d[:, :2]*0.27, arr_col, loop=False)

# Clean up
cap.release()
cv2.destroyAllWindows()